In [1]:
# 1. Instalar la librería de YOLO
!pip install ultralytics

# 2. Importar las herramientas necesarias
from ultralytics import YOLO
from google.colab.patches import cv2_imshow
import cv2


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 5.6 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
# Descomprime el archivo ZIP en una carpeta local llamada 'mi_dataset'
!unzip -q "/content/drive/MyDrive/Residencia/DatasetResidencia-2808.v1-version1-definitivaresidencia.yolov11.zip" -d /content/mi_dataset
# Código para verificar que las 11,000 imágenes están listas
import os
ruta_local = '/content/mi_dataset'

# Si al comprimir se creó una carpeta interna, ajustamos la ruta automáticamente
contenidos = os.listdir(ruta_local)
if len(contenidos) == 1 and os.path.isdir(os.path.join(ruta_local, contenidos[0])):
    ruta_local = os.path.join(ruta_local, contenidos[0])

total_archivos = len(os.listdir(ruta_local))
print(f"¡Proceso completado con éxito!")
print(f"Se han descomprimido {total_archivos} imágenes en el almacenamiento rápido de Colab.")

replace /content/mi_dataset/data.yaml? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace /content/mi_dataset/data.yaml? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
¡Proceso completado con éxito!
Se han descomprimido 6 imágenes en el almacenamiento rápido de Colab.


In [6]:
import os
import yaml

# Busca automáticamente cualquier archivo llamado 'data.yaml' en /content/mi_dataset
yaml_encontrado = None
for raiz, dirs, archivos in os.walk('/content/mi_dataset'):
    if 'data.yaml' in archivos:
        yaml_encontrado = os.path.join(raiz, 'data.yaml')
        break

if yaml_encontrado:
    print(f"Archivo encontrado en: {yaml_encontrado}")

    # 1. Leer el archivo yaml actual
    with open(yaml_encontrado, 'r') as f:
        datos_yaml = yaml.safe_load(f)

    # 2. Asignar la ruta raíz según la carpeta donde realmente está el YAML
    carpeta_raiz = os.path.dirname(yaml_encontrado)
    datos_yaml['path'] = carpeta_raiz
    datos_yaml['train'] = 'train/images'
    datos_yaml['val'] = 'valid/images'
    datos_yaml['test'] = 'test/images'

    # 3. Guardar los cambios
    with open(yaml_encontrado, 'w') as f:
        yaml.dump(datos_yaml, f, default_flow_style=False)

    print("¡Archivo data.yaml corregido con éxito!")
else:
    print("No se encontró ningún archivo 'data.yaml' en /content/mi_dataset. Verifica si el zip se descomprimió bien.")

Archivo encontrado en: /content/mi_dataset/data.yaml
¡Archivo data.yaml corregido con éxito!


In [ ]:
from ultralytics import YOLO

# 1. Cargamos un modelo preentrenado base (la versión 'n' es la más rápida para empezar)
model = YOLO('yolo11m.pt')

# 2. Iniciamos el entrenamiento apuntando al archivo yaml que se descomprimió en Colab
results = model.train(
    data='/content/mi_dataset/data.yaml', # Ruta al archivo yaml local
    epochs=200,                                                 # Número de vueltas de entrenamientoa
    patience = 20,
    imgsz=640,                                                 # Tamaño estándar de redimensión de imágenes
    batch=16,                                                  # Lotes pequeños para no saturar la GPU de Colab
    device=0,                                                 # Le indica que use la GPU T4
    project='/content/drive/MyDrive/Residencia/entrenamientos',  # Se guarda en Drive
    name='experimento_yolo11m'                                      # Nombre de la carpeta

)

Ultralytics 8.4.132 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/mi_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=experimento_yolo11m, nbs=64, n

In [7]:
from ultralytics import YOLO

# 1. Cargar el último punto de control (last.pt) desde tu Google Drive
model = YOLO('/content/drive/MyDrive/Residencia/entrenamientos/experimento_yolo11m/weights/last.pt')

# 2. Reanudar el entrenamiento
results = model.train(resume=True)

Ultralytics 8.4.135 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/mi_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=814, mixup=0.0, mode=train, model=/content/drive/MyDrive/Residencia/entrenamientos/experimento_yolo11m/weights/last.pt, moment

KeyboardInterrupt: 

In [ ]:
from ultralytics import YOLO

# 1. Cargas el mejor modelo que obtuviste en la ronda anterior
model = YOLO('/content/runs/detect/train-2/weights/best.pt')

# 2. Inicias un nuevo entrenamiento (YOLO creará de forma automática la carpeta 'train2')
results = model.train(
    data='/content/mi_dataset/datasetBalanceado-v3/data.yaml',
    epochs=10,   # Aquí indicas cuántas épocas EXTRAS quieres sumarle
    imgsz=640,
    batch=16,
    device=0
)

Ultralytics 8.4.66 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/mi_dataset/datasetBalanceado-v3/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/runs/detect/train-2/weights/best.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-3, nbs=64, nms=False, opset=None, optimize=F

In [9]:
# Copia la carpeta de progreso actual a tu Drive de forma permanente
!cp -r  /content/runs/detect/val /content/drive/MyDrive/Residencia/entrenamientos
print("¡A salvo! Todo lo generado hasta el momento se ha copiado en tu Google Drive.")

¡A salvo! Todo lo generado hasta el momento se ha copiado en tu Google Drive.


In [8]:
from ultralytics import YOLO

# 1. Cargar el modelo guardado en best.pt
model = YOLO('/content/drive/MyDrive/Residencia/entrenamientos/experimento_yolo11m/weights/best.pt')  # Ajusta la ruta a tu archivo best.pt

# 2. Evaluar sobre el conjunto de validación o test
metrics = model.val(data="/content/mi_dataset/data.yaml", plots=True)

# 3. Mostrar resultados
print(metrics)

Ultralytics 8.4.135 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11m summary (fused): 126 layers, 20,057,788 parameters, 0 gradients, 67.9 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 463.9±401.6 MB/s, size: 83.8 KB)
val: Scanning /content/mi_dataset/valid/labels.cache... 3672 images, 12 backgrounds, 74 corrupt: 100% ━━━━━━━━━━━━ 3672/3672 375.6Mit/s 0.0s
val: /content/mi_dataset/valid/images/Apple-tree-Pictures_jpg.rf.3203b29be42196f0624b33a3e23d1b4d.jpg: ignoring corrupt image/label: labels mix segment and detection rows
val: /content/mi_dataset/valid/images/Apple-tree-Wallpaper_jpg.rf.68bff78d07080f61247623d71c6ac6d2.jpg: ignoring corrupt image/label: labels mix segment and detection rows
val: /content/mi_dataset/valid/images/Apples-In-Tree-1-_jpg.rf.9531d8d68229275834d0f96025f29c37.jpg: ignoring corrupt image/label: labels mix segment and detection rows
val: /content/mi_dataset/valid/images/CT13_png.rf.db8e19235507dc927c07e1eba1134562.jpg: ignorin

In [ ]:
from ultralytics import YOLO
import time

# -----------------------------
# CONFIGURACIÓN
# -----------------------------
MODEL_PATH = "yolov8l.pt"
DATA_YAML = "/content/mi_dataset/datasetBalanceado-v3/data.yaml"                           # Ruta a tu dataset YAML
FPS_ITERATIONS = 10                                # Número de veces que se medirá FPS para promedio
TEST_IMAGE = "/content/mi_dataset/datasetBalanceado-v3/train/images/1_jpeg.rf.5914e21362216a12e287b60f06ab59b9.jpg"  # Imagen de prueba para FPS


# -----------------------------
# INFO DEL MODELO
# -----------------------------
layers, params, grads, gflops = model.info()
params_m = params / 1e6

# -----------------------------
# MÉTRICAS PRINCIPALES
# -----------------------------
res = metrics.results_dict
ap50 = res['metrics/mAP50(B)']
ap50_95 = res['metrics/mAP50-95(B)']
precision = res['metrics/precision(B)']
recall = res['metrics/recall(B)']

# Calcular F1-score
f1_score = 2 * (precision * recall) / (precision + recall)

# -----------------------------
# FPS (Frames por segundo)
# -----------------------------
t_total = 0
for _ in range(FPS_ITERATIONS):
    t0 = time.time()
    _ = model(TEST_IMAGE)
    t_total += (time.time() - t0)
fps = FPS_ITERATIONS / t_total

# -----------------------------
# IMPRIMIR RESULTADOS
# -----------------------------
print("\n========== MÉTRICAS DEL MODELO ==========")
print(f"Params(M): {params_m:.2f}")
print(f"GFLOPs: {gflops}")
print(f"FPS (bs=1): {fps:.2f}")
print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"F1-score: {f1_score:.3f}")
print(f"APval50: {ap50:.3f}")
print(f"APval50-95: {ap50_95:.3f}")
print("========================================\n")


Model summary (fused): 113 layers, 43,629,738 parameters, 0 gradients, 164.9 GFLOPs

image 1/1 /content/mi_dataset/datasetBalanceado-v3/train/images/1_jpeg.rf.5914e21362216a12e287b60f06ab59b9.jpg: 640x640 2 Pineapples, 57.0ms
Speed: 3.3ms preprocess, 57.0ms inference, 1.4ms postprocess per image at shape (1, 3, 640, 640)

image 1/1 /content/mi_dataset/datasetBalanceado-v3/train/images/1_jpeg.rf.5914e21362216a12e287b60f06ab59b9.jpg: 640x640 2 Pineapples, 42.8ms
Speed: 3.1ms preprocess, 42.8ms inference, 1.4ms postprocess per image at shape (1, 3, 640, 640)

image 1/1 /content/mi_dataset/datasetBalanceado-v3/train/images/1_jpeg.rf.5914e21362216a12e287b60f06ab59b9.jpg: 640x640 2 Pineapples, 42.5ms
Speed: 2.6ms preprocess, 42.5ms inference, 1.3ms postprocess per image at shape (1, 3, 640, 640)

image 1/1 /content/mi_dataset/datasetBalanceado-v3/train/images/1_jpeg.rf.5914e21362216a12e287b60f06ab59b9.jpg: 640x640 2 Pineapples, 41.4ms
Speed: 2.3ms preprocess, 41.4ms inference, 1.2ms postproc

In [ ]:
# Copia la carpeta de progreso actual a tu Drive de forma permanente
!cp -r /content/runs/detect/val /content/drive/MyDrive/Datasets-Memoria/pruebayolov8l-95-val
print("¡A salvo! Todo lo generado hasta el momento se ha copiado en tu Google Drive.")

¡A salvo! Todo lo generado hasta el momento se ha copiado en tu Google Drive.
